# Pha P'.0 — Ablation "dequantization" cổ điển (trước khi vào lượng tử)

Mục đích (xem `docs/NHIP2_GUIDE.md` mục 1–2): tách bạch "lợi ích từ dạng hàm
Fourier" (Schuld, Sweke & Meyer, *Phys. Rev. A* 103, 032430, 2021 — mạch
data re-uploading tương đương 1 chuỗi Fourier riêng phần của dữ liệu) khỏi
"lợi ích từ bản chất lượng tử" (entanglement, Hilbert space lớn), TRƯỚC khi
cài PennyLane/viết mạch lượng tử thật.

**Lịch sử sửa (2 vòng chạy trước đã phát hiện + sửa):**
1. Bản đầu (`encode_w/encode_b` học được): DV-bound sụp đổ về nghiệm tầm
   thường `T=const` ngay epoch 1 (val loss ~0 suốt quá trình) — MSE của
   `Fourier_classical` gần như hằng số ở mọi N, không dùng được.
2. Sửa `encode_w/encode_b` thành cố định (Random Fourier Features chuẩn,
   Rahimi & Recht 2007) — vẫn sụp đổ (dù chậm hơn), kể cả tăng learning rate
   50 lần.
3. **Bản này:** thêm `donsker_varadhan_loss_ema()` (hiệu chỉnh gradient chuẩn
   của MINE, Belghazi et al. 2018 mục 3.2) + tăng `windows_per_batch` 32→128
   (giảm nhiễu gradient). Kiểm tra nhanh ở quy mô nhỏ cho tín hiệu tốt hơn
   (weight norm không suy biến về 0 sau 12 epoch) nhưng CHƯA chắc chắn hội tụ
   — đây là lần chạy đầy đủ để xác nhận.

**Không tính lại KSG** — tái dùng đúng số liệu KSG đã có trong
`phase_r_grid_summary.csv`/`phase_r2_periodic_grid_summary.csv`, chỉ train +
đánh giá thêm đúng 1 estimator mới trên đúng tập test đã đóng băng.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import yaml, time, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from pathlib import Path

from pqrst.data.synthetic.corpus import load_corpus
from pqrst.estimators.mine.amortized import train_amortized, AmortizedTrainConfig, AmortizedTEEstimator
from pqrst.estimators.classical_fourier import FourierFeatureStatisticsNetwork
from pqrst.evaluation.grid import evaluate_estimators_on_grid, summarize_grid

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
cfg = yaml.safe_load(open(BASE / 'configs' / 'mine' / 'amortized.yaml', encoding='utf-8'))
train_cfg = cfg['train']
N_HARMONICS = 4

n_params_fourier = sum(p.numel() for p in FourierFeatureStatisticsNetwork(N_HARMONICS).parameters())
print(f'FourierFeatureStatisticsNetwork: {n_params_fourier} tham so hoc duoc (chi lop doc ra; '
      f'n_harmonics={N_HARMONICS})')

## 1. Dữ liệu tuyến tính (VAR) — train + đánh giá

In [ ]:
corpus_dir = BASE / 'data' / 'interim' / 'synthetic_corpus'
train_windows = load_corpus(str(corpus_dir / 'train.npz'))
val_windows = load_corpus(str(corpus_dir / 'val.npz'))
test_windows_linear = load_corpus(str(corpus_dir / 'test.npz'))
print(f'Linear: train={len(train_windows)}, val={len(val_windows)}, test={len(test_windows_linear)}')

In [ ]:
config_fourier = AmortizedTrainConfig(
    learning_rate=train_cfg['learning_rate'],
    windows_per_batch=128,  # SUA (xem docs/NHIP2_GUIDE.md muc 2.1): 32 (goc, giong
                            # T_phi) lam DV bound sup do ve nghiem tam thuong T=const
                            # cho mo hinh Fourier nho - tang batch de giam nhieu gradient.
    max_epochs=train_cfg['max_epochs'],
    patience=train_cfg['patience'],
    eval_n_shuffles=train_cfg['eval_n_shuffles'],
    final_estimate_last_k_epochs=train_cfg['final_estimate_last_k_epochs'],
    seed=train_cfg['seed'],
    standardize=False,  # khop dung cach phi_amortized.pt/phi_amortized_periodic.pt da train,
                         # de so sanh apples-to-apples voi phase_r_grid_summary.csv
)

start = time.time()
est_fourier_linear, hist_fourier_linear = train_amortized(
    train_windows, val_windows, config_fourier,
    model_factory=lambda: FourierFeatureStatisticsNetwork(N_HARMONICS),
    use_ema_correction=True, ema_momentum=0.01,  # SUA: hieu chinh gradient chuan cua
                                                  # MINE (losses.py) - loss thuong da
                                                  # xac nhan sup do, xem markdown dau bai.
)
print(f'Time: {time.time()-start:.1f}s, epochs run: {len(hist_fourier_linear["train_loss_history"])}, '
      f'final_val_loss={hist_fourier_linear["final_val_loss"]:.5f}, '
      f'readout_weight_norm={est_fourier_linear.model.readout.weight.norm().item():.4f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(hist_fourier_linear['train_loss_history'], label='Train Loss')
plt.plot(hist_fourier_linear['val_loss_history'], label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('-DV bound (EMA-corrected)'); plt.legend()
plt.title('Fourier classical (dequantization ablation) - Linear VAR - loss curve')
out_dir = BASE / 'results' / 'figures'; out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / 'phase_p2_fourier_linear_loss.png', dpi=100)
plt.show()
print('KIEM TRA TRUOC KHI DI TIEP: neu readout_weight_norm o tren gan 0 (<0.01) hoac',
      'loss van dang giam deu ve gan 0 khong chung lai - MO HINH VAN SUP DO, dung',
      'lai bao cho Claude thay vi chay het notebook.')

In [ ]:
raw_fourier_linear = evaluate_estimators_on_grid(
    {'Fourier_classical': est_fourier_linear}, test_windows_linear, progress=True)
summary_fourier_linear = summarize_grid(raw_fourier_linear)
summary_fourier_linear.head()

## 2. Dữ liệu phi tuyến (periodic coupling) — train + đánh giá

In [ ]:
corpus_dir_periodic = BASE / 'data' / 'interim' / 'synthetic_corpus_periodic'
train_windows_p = load_corpus(str(corpus_dir_periodic / 'train.npz'))
val_windows_p = load_corpus(str(corpus_dir_periodic / 'val.npz'))
test_windows_periodic = load_corpus(str(corpus_dir_periodic / 'test.npz'))
print(f'Periodic: train={len(train_windows_p)}, val={len(val_windows_p)}, test={len(test_windows_periodic)}')

In [ ]:
start = time.time()
est_fourier_periodic, hist_fourier_periodic = train_amortized(
    train_windows_p, val_windows_p, config_fourier,
    model_factory=lambda: FourierFeatureStatisticsNetwork(N_HARMONICS),
    use_ema_correction=True, ema_momentum=0.01,
)
print(f'Time: {time.time()-start:.1f}s, epochs run: {len(hist_fourier_periodic["train_loss_history"])}, '
      f'final_val_loss={hist_fourier_periodic["final_val_loss"]:.5f}, '
      f'readout_weight_norm={est_fourier_periodic.model.readout.weight.norm().item():.4f}')

In [ ]:
raw_fourier_periodic = evaluate_estimators_on_grid(
    {'Fourier_classical': est_fourier_periodic}, test_windows_periodic, progress=True)
summary_fourier_periodic = summarize_grid(raw_fourier_periodic)
summary_fourier_periodic.head()

## 3. Gộp với KSG/Amortized đã có sẵn (KHÔNG tính lại) + so sánh

In [ ]:
tables_dir = BASE / 'results' / 'tables'

df_linear_existing = pd.read_csv(tables_dir / 'phase_r_grid_summary.csv')
df_linear_existing = df_linear_existing[df_linear_existing['estimator'].isin(['KSG', 'Amortized'])].copy()
summary_fourier_linear['estimator'] = 'Fourier_classical'
df_linear = pd.concat([df_linear_existing, summary_fourier_linear], ignore_index=True)
df_linear['dataset'] = 'Linear VAR'

df_periodic_existing = pd.read_csv(tables_dir / 'phase_r2_periodic_grid_summary.csv')
df_periodic_existing = df_periodic_existing[df_periodic_existing['estimator'].isin(['KSG', 'Amortized'])].copy()
summary_fourier_periodic['estimator'] = 'Fourier_classical'
df_periodic = pd.concat([df_periodic_existing, summary_fourier_periodic], ignore_index=True)
df_periodic['dataset'] = 'Periodic Coupling'

df_all = pd.concat([df_linear, df_periodic], ignore_index=True)
df_all.to_csv(tables_dir / 'phase_p2_dequantization_summary.csv', index=False)
print(f'Da luu {len(df_all)} dong -> phase_p2_dequantization_summary.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'KSG': '#7f8c8d', 'Amortized': '#2980b9', 'Fourier_classical': '#e67e22'}
for ax, ds in zip(axes, ['Linear VAR', 'Periodic Coupling']):
    sub = df_all[df_all['dataset'] == ds]
    for est, color in colors.items():
        g = sub[sub['estimator'] == est].groupby('n_samples')['mse'].mean().reset_index()
        ax.plot(g['n_samples'], g['mse'], marker='o', label=est, color=color)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('N'); ax.set_ylabel('MSE'); ax.set_title(ds); ax.legend()
fig.suptitle('Dequantization ablation: MSE vs N - KSG vs Amortized (MLP) vs Fourier classical (33 params)', y=1.02)
plt.tight_layout()
plt.savefig(out_dir / 'phase_p2_dequantization_mse.png', dpi=110, bbox_inches='tight')
plt.show()

## 4. Kết luận (tính từ số liệu, không đoán trước)

In [ ]:
for ds in ['Linear VAR', 'Periodic Coupling']:
    sub = df_all[df_all['dataset'] == ds]
    piv = sub.groupby(['n_samples', 'estimator'])['mse'].mean().unstack()
    n_fourier_beats_amortized = int((piv['Fourier_classical'] < piv['Amortized']).sum())
    n_total = len(piv)
    # Kiem tra co con dau hieu sup do khong: neu MSE giong het nhau (do lech <1e-6)
    # o moi N thi van la collapse, khong phai ket qua dung duoc.
    mse_range_fourier = piv['Fourier_classical'].max() - piv['Fourier_classical'].min()
    print(f'=== {ds} ===')
    print(piv)
    print(f'-> Fourier_classical thang Amortized (MLP) ve MSE o {n_fourier_beats_amortized}/{n_total} gia tri N.')
    print(f'-> Do bien thien MSE cua Fourier_classical qua cac N: {mse_range_fourier:.6f} '
          f'({"CANH BAO: qua nho, co the van con sup do" if mse_range_fourier < 1e-5 else "OK, khac nhau co nghia theo N"})')
    print()

print('Doc ket qua theo docs/NHIP2_GUIDE.md muc 1:')
print('- Neu KHONG con canh bao sup do o tren: so sanh MSE that su co nghia.')
print('  + Fourier_classical thang/ngang Amortized -> tin hieu tot co the den tu DANG')
print('    HAM Fourier, chua chac can luong tu thuc.')
print('  + Fourier_classical thua ro -> can luong tu thuc (entanglement) manh hon.')
print('- Neu VAN con canh bao sup do: ablation nay CHUA co cau tra loi (khong dung so')
print('  lieu de ket luan), can bao Claude de quyet dinh buoc tiep (dau tu on dinh hoa')
print('  MINE training sau hon, hoac chap nhan ghi nhan chinh hien tuong sup do nay).')